# E1 — YOLO26s-P2 trên DataSet2

Thí nghiệm ablation đầu tiên cho đề tài nhận diện người đi xe máy không đội mũ bảo hiểm.

- Baseline tham chiếu: YOLO26s với `yolo26s.pt`.
- E1 duy nhất thay đổi architecture: YOLO26s + P2 Detection Head.
- Không thêm attention, DySample, iAFF, SEAM, MultiSEAM, custom loss hoặc tuning khác.
- Dataset, class order và training protocol được giữ nguyên như notebook `Train3.ipynb`.

Notebook này là bản mới, không ghi đè notebook baseline.

In [ ]:
# Google Colab: cài đúng các dependency mà notebook baseline đang dùng.
!pip install -U ultralytics
!pip install -q gdown
!apt-get update -qq
!apt-get install -y unrar

In [ ]:
# Thiết lập dataset trên Google Colab — giữ nguyên dataset của baseline.
from google.colab import drive
import os
import gdown

drive.mount('/content/drive', force_remount=True)

!mkdir -p /content/DetectedHelmets/DataSet2
!mkdir -p /content/DetectedHelmets/Code
%cd /content/DetectedHelmets

if os.path.exists('dataset.rar'):
    os.remove('dataset.rar')

file_id = '1LUUQLl5F3aczuny4As_PcTtqLIWTmZSJ'
url = f'https://drive.google.com/uc?id={file_id}'
output = 'dataset.rar'
gdown.download(url, output, quiet=False, fuzzy=True)

print('Đang giải nén dataset hiện tại...')
!unrar x -o+ {output} /content/DetectedHelmets/DataSet2/
print('Tải xuống và giải nén dataset hoàn tất.')

In [ ]:
# Ưu tiên bản dataset trên Google Drive nếu đã có, giống notebook baseline.
drive_dataset_path = '/content/drive/MyDrive/DataSet2'
local_dataset_path = '/content/DetectedHelmets/DataSet2'

if os.path.exists(drive_dataset_path):
    if os.path.islink(local_dataset_path) or os.path.exists(local_dataset_path):
        !rm -rf {local_dataset_path}
    !ln -s "{drive_dataset_path}" "{local_dataset_path}"
    print(f'Đã liên kết dataset: {local_dataset_path} -> {drive_dataset_path}')
else:
    print(f'Không tìm thấy đường dẫn dataset trên Drive: {drive_dataset_path}')

## 1. Environment và GPU check

In [ ]:
import torch
import ultralytics

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch: {torch.__version__}')

# Benchmark này bắt buộc chạy trên GPU device=0, không tự động chuyển sang CPU.
assert torch.cuda.is_available(), "GPU chưa được bật; hãy bật GPU runtime trong Google Colab trước khi train."

## 2. Dataset, class order và output paths

In [ ]:
import os
from pathlib import Path
import yaml

CURRENT_DIR = Path(os.getcwd()).resolve()
ROOT_DIR = CURRENT_DIR
while ROOT_DIR.name in ['Train', 'Code'] and ROOT_DIR.parent != ROOT_DIR:
    ROOT_DIR = ROOT_DIR.parent

# Giữ nguyên DATA_YAML_PATH của Train3.ipynb baseline.
DATA_YAML_PATH = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'data.yaml'
MODEL_NAME = 'yolo26s.pt'
P2_MODEL_CONFIG = 'yolo26s-p2.yaml'

OUTPUT_DIR = Path('/content/drive/MyDrive/DetectedHelmets/runs/train')
EXPERIMENT_NAME = 'helmet_yolo26s_p2_dataset2'
BASELINE_EXPERIMENT_NAME = 'helmet_yolo26s_dataset2'
EXPERIMENT_DIR = OUTPUT_DIR / EXPERIMENT_NAME
BASELINE_BEST_MODEL_PATH = OUTPUT_DIR / BASELINE_EXPERIMENT_NAME / 'weights' / 'best.pt'

print(f'Project root       : {ROOT_DIR}')
print(f'DATA_YAML_PATH     : {DATA_YAML_PATH}')
print(f'Baseline model     : {MODEL_NAME}')
print(f'P2 architecture    : {P2_MODEL_CONFIG}')
print(f'Experiment output  : {EXPERIMENT_DIR}')
print(f'Baseline best.pt   : {BASELINE_BEST_MODEL_PATH}')

if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy data.yaml của baseline: {DATA_YAML_PATH}')

with open(DATA_YAML_PATH, 'r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

class_names = data_config.get('names', [])
if isinstance(class_names, dict):
    class_names = [class_names[key] for key in sorted(class_names, key=lambda item: int(item))]
CLASS_NAMES = list(class_names)

EXPECTED_CLASS_NAMES = [
    'motorcycle',
    'rider_with_helmet',
    'rider_without_helmet',
]

if data_config.get('nc') != 3:
    raise ValueError(f'Baseline dataset phải có nc=3, nhận được: {data_config.get("nc")}')
if CLASS_NAMES != EXPECTED_CLASS_NAMES:
    raise ValueError(
        'Class order không đúng baseline. '
        f'Expected={EXPECTED_CLASS_NAMES}, actual={CLASS_NAMES}'
    )

print('\n--- Dataset configuration ---')
print(f'nc    : {data_config.get("nc")}')
print(f'names : {CLASS_NAMES}')
print('Dataset và class labels đã được kiểm tra, không thay đổi.')

## 3. Khởi tạo architecture chính thức YOLO26s-P2

Cell này cố ý không dùng `YOLO("yolo26s.pt")` làm architecture training. YAML P2 được khởi tạo trước, sau đó mới transfer weights tương thích từ `yolo26s.pt`.

In [ ]:
from ultralytics import YOLO

try:
    model = YOLO("yolo26s-p2.yaml")
except Exception as exc:
    raise RuntimeError(
        'Ultralytics hiện tại không cung cấp architecture yolo26s-p2.yaml. '
        'Không tự tạo YAML/custom head; hãy dùng phiên bản Ultralytics có architecture chính thức này.'
    ) from exc

model.load("yolo26s.pt")

if getattr(model, 'task', None) != 'detect':
    raise ValueError(f'Experiment phải là object detection, nhận task={getattr(model, "task", None)!r}')

print(f'Đã khởi tạo {P2_MODEL_CONFIG} và load pretrained weights từ {MODEL_NAME}.')
print(f'Model task: {model.task}')
print('Model info trước training:')
model.info(verbose=True, imgsz=640)

## 4. Training E1

Giữ nguyên benchmark protocol của baseline: 50 epoch, 640px, batch 16, device 0, workers 4, optimizer auto, seed 0, deterministic và patience 15. Output được tách riêng khỏi baseline.

In [ ]:
train_results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    optimizer='auto',
    seed=0,
    deterministic=True,
    patience=15,
    project=str(OUTPUT_DIR),
    name="helmet_yolo26s_p2_dataset2",
    save=True,
    save_period=10,
    plots=True,
    exist_ok=False,
)

## 5. Load đúng best checkpoint của E1

In [ ]:
BEST_MODEL_PATH = EXPERIMENT_DIR / 'weights' / 'best.pt'

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy best.pt của E1: {BEST_MODEL_PATH}')

best_model = YOLO(BEST_MODEL_PATH)
if getattr(best_model, 'task', None) != 'detect':
    raise ValueError(f'best.pt phải là object detection, nhận task={getattr(best_model, "task", None)!r}')

print(f'Đã load đúng checkpoint: {BEST_MODEL_PATH}')

## 6. Validation trên split='val'

In [ ]:
import numpy as np
import pandas as pd

def safe_f1(precision_value, recall_value):
    denominator = precision_value + recall_value
    return 2.0 * precision_value * recall_value / denominator if denominator else 0.0

def extract_metrics(split_metrics, split_name):
    total_precision = float(split_metrics.box.mp)
    total_recall = float(split_metrics.box.mr)
    total_map50 = float(split_metrics.box.map50)
    total_map5095 = float(split_metrics.box.map)
    total_f1 = safe_f1(total_precision, total_recall)

    by_class = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        precision_value, recall_value, ap50_value, ap5095_value = split_metrics.box.class_result(class_id)
        precision_value = float(precision_value)
        recall_value = float(recall_value)
        ap50_value = float(ap50_value)
        ap5095_value = float(ap5095_value)
        by_class[class_name] = {
            'Precision': precision_value,
            'Recall': recall_value,
            'F1': safe_f1(precision_value, recall_value),
            'AP50': ap50_value,
            'AP50-95': ap5095_value,
        }

    print(f'--- Validation metrics ({split_name}) ---')
    print(f'Precision : {total_precision:.4f}')
    print(f'Recall    : {total_recall:.4f}')
    print(f'F1        : {total_f1:.4f}')
    print(f'mAP50     : {total_map50:.4f}')
    print(f'mAP50-95  : {total_map5095:.4f}')

    print('\n--- Metrics theo từng class ---')
    for class_name, values in by_class.items():
        print(
            f'{class_name:24s} | '
            f'Precision={values["Precision"]:.4f} | '
            f'Recall={values["Recall"]:.4f} | '
            f'F1={values["F1"]:.4f} | '
            f'AP50={values["AP50"]:.4f} | '
            f'AP50-95={values["AP50-95"]:.4f}'
        )

    return {
        'Precision': total_precision,
        'Recall': total_recall,
        'F1': total_f1,
        'mAP50': total_map50,
        'mAP50-95': total_map5095,
        'by_class': by_class,
    }

metrics = best_model.val(
    data=str(DATA_YAML_PATH),
    split='val',
    imgsz=640,
    device=0,
    plots=True,
)
validation_summary = extract_metrics(metrics, 'validation')

no_helmet_metrics = validation_summary['by_class']['rider_without_helmet']
print('\n--- Class quan trọng: rider_without_helmet ---')
for metric_name in ('Precision', 'Recall', 'F1', 'AP50', 'AP50-95'):
    print(f'{metric_name:10s}: {no_helmet_metrics[metric_name]:.4f}')

## 7. Test set disabled during E1

Test set được để dành cho Proposed Model cuối cùng; E1 chỉ sử dụng validation.

In [ ]:
# E1 ablation: không chạy test set. Test chỉ dùng sau khi chốt Proposed Model cuối cùng.
test_metrics = None
test_summary = None
print('Test evaluation đã được vô hiệu hóa trong E1; chỉ validation được sử dụng.')

## 8. Hiển thị results, confusion matrix và curves

Ultralytics phiên bản hiện tại dùng prefix `Box` cho các curve của object detection.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

PLOT_SPECS = [
    ('results.png', 'Training results'),
    ('confusion_matrix.png', 'Confusion matrix'),
    ('confusion_matrix_normalized.png', 'Normalized confusion matrix'),
    ('BoxPR_curve.png', 'Box Precision-Recall curve'),
    ('BoxF1_curve.png', 'Box F1 curve'),
    ('BoxP_curve.png', 'Box Precision curve'),
    ('BoxR_curve.png', 'Box Recall curve'),
]

def find_experiment_plot(filename):
    candidates = list(EXPERIMENT_DIR.rglob(filename)) if EXPERIMENT_DIR.exists() else []
    if not candidates:
        return None
    return max(candidates, key=lambda path: path.stat().st_mtime)

for filename, title in PLOT_SPECS:
    plot_path = find_experiment_plot(filename)
    if plot_path is None:
        print(f'Không tìm thấy {filename} trong {EXPERIMENT_DIR}.')
        continue

    plt.figure(figsize=(16, 10) if filename == 'results.png' else (10, 8))
    plt.imshow(Image.open(plot_path))
    plt.axis('off')
    plt.title(f'{title} — {plot_path.parent.name}')
    plt.show()

## 9. Error analysis từ confusion matrix

Ultralytics lưu confusion matrix detection theo quy ước `matrix[predicted, actual]`, với hàng/cột cuối là background. Vì vậy các lỗi được in theo hướng `actual -> predicted`. Nếu API không trả object matrix trực tiếp, notebook chỉ hiển thị plot và không tự suy đoán số liệu.

In [ ]:
def get_confusion_matrix_array(split_metrics):
    confusion_object = getattr(split_metrics, 'confusion_matrix', None)
    matrix = getattr(confusion_object, 'matrix', None)
    if matrix is None:
        return None
    matrix = np.asarray(matrix)
    expected_shape = (len(CLASS_NAMES) + 1, len(CLASS_NAMES) + 1)
    return matrix if matrix.shape == expected_shape else None

confusion_matrix_array = get_confusion_matrix_array(metrics)

if confusion_matrix_array is None:
    print('Không đọc được confusion matrix object trực tiếp từ metrics API.')
    print('Không tạo số liệu lỗi nhân tạo; hãy dùng confusion_matrix.png đã hiển thị ở cell trước.')
else:
    confusion_labels = CLASS_NAMES + ['background']
    confusion_table = pd.DataFrame(
        confusion_matrix_array,
        index=[f'predicted:{label}' for label in confusion_labels],
        columns=[f'actual:{label}' for label in confusion_labels],
    )
    print('--- Raw confusion matrix (hàng=predicted, cột=actual) ---')
    display(confusion_table)

    label_to_index = {label: index for index, label in enumerate(confusion_labels)}

    def error_count(actual_label, predicted_label):
        return float(
            confusion_matrix_array[
                label_to_index[predicted_label],
                label_to_index[actual_label],
            ]
        )

    error_rows = [
        {
            'Error': 'rider_without_helmet -> background',
            'Count': error_count('rider_without_helmet', 'background'),
            'Interpretation': 'Ground-truth rider không đội mũ bị bỏ sót (false negative).',
        },
        {
            'Error': 'rider_without_helmet -> rider_with_helmet',
            'Count': error_count('rider_without_helmet', 'rider_with_helmet'),
            'Interpretation': 'Ground-truth rider không đội mũ bị nhận nhầm thành rider đội mũ.',
        },
        {
            'Error': 'background -> rider_without_helmet',
            'Count': error_count('background', 'rider_without_helmet'),
            'Interpretation': 'Mô hình dự đoán rider không đội mũ ở vùng không có ground-truth object (false positive).',
        },
    ]
    print('--- Các lỗi mục tiêu ---')
    display(pd.DataFrame(error_rows))

## 10. Prediction examples trên validation set

Các ảnh ứng viên được lấy ngẫu nhiên từ validation set với seed cố định, sau đó ưu tiên ảnh có nhiều prediction `rider_without_helmet`, nhiều box nhỏ và nhiều rider để dễ kiểm tra định tính.

In [ ]:
import math
import random
from pathlib import Path

valid_img_dir = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'valid' / 'images'
valid_images = sorted(
    [
        str(path)
        for pattern in ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
        for path in valid_img_dir.glob(pattern)
    ]
)

if not valid_images:
    raise FileNotFoundError(f'Không tìm thấy ảnh validation trong: {valid_img_dir}')

rng = random.Random(0)
candidate_images = rng.sample(valid_images, min(20, len(valid_images)))
candidate_results = best_model.predict(
    source=candidate_images,
    conf=0.25,
    imgsz=640,
    device=0,
    save=False,
    verbose=False,
)

no_helmet_class_id = CLASS_NAMES.index('rider_without_helmet')

def result_priority(result):
    if result.boxes is None or len(result.boxes) == 0:
        return (0, 0, 0)

    class_ids = result.boxes.cls.detach().cpu().numpy().astype(int)
    xyxy = result.boxes.xyxy.detach().cpu().numpy()
    height, width = result.orig_shape
    image_area = max(float(height * width), 1.0)
    areas = ((xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])) / image_area

    no_helmet_count = int(np.sum(class_ids == no_helmet_class_id))
    small_box_count = int(np.sum(areas < 0.03))
    total_box_count = int(len(class_ids))
    return (no_helmet_count, small_box_count, total_box_count)

ranked_results = sorted(
    candidate_results,
    key=result_priority,
    reverse=True,
)
selected_results = ranked_results[:min(4, len(ranked_results))]

ncols = 2
nrows = max(1, math.ceil(len(selected_results) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 6 * nrows))
axes = np.atleast_1d(axes).ravel()

for index, result in enumerate(selected_results):
    rendered_rgb = result.plot(conf=True, labels=True, boxes=True)[..., ::-1]
    axes[index].imshow(rendered_rgb)
    axes[index].axis('off')
    axes[index].set_title(Path(result.path).name, fontsize=10)

for axis in axes[len(selected_results):]:
    axis.axis('off')

plt.tight_layout()
plt.show()

## 11. Model complexity và inference speed

Đo trên checkpoint E1 `best.pt`, cùng `imgsz=640` và `device=0`; inference có warm-up trước khi tính trung bình.

In [ ]:
import time

parameters = sum(parameter.numel() for parameter in best_model.model.parameters())
model_info = best_model.info(verbose=True, imgsz=640)

def extract_gflops(info):
    if isinstance(info, (tuple, list)) and len(info) >= 4:
        try:
            return float(info[-1])
        except (TypeError, ValueError):
            pass
    if isinstance(info, dict):
        for key in ('GFLOPs', 'gflops', 'flops'):
            if key in info:
                try:
                    return float(info[key])
                except (TypeError, ValueError):
                    pass
    return None

gflops = extract_gflops(model_info)
model_size_mb = BEST_MODEL_PATH.stat().st_size / (1024 ** 2)

speed_images = valid_images[:min(50, len(valid_images))]
if not speed_images:
    valid_img_dir = ROOT_DIR / 'DataSet2' / 'DataSet2' / 'valid' / 'images'
    speed_images = [str(path) for path in valid_img_dir.glob('*.*')][:50]

if not speed_images:
    raise FileNotFoundError('Không có ảnh validation để đo inference speed.')

for image_path in speed_images[:min(2, len(speed_images))]:
    best_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)

inference_times = []
for image_path in speed_images:
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start_time = time.perf_counter()
    best_model.predict(source=image_path, imgsz=640, device=0, save=False, verbose=False)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_times.append(time.perf_counter() - start_time)

inference_ms = float(np.mean(inference_times) * 1000.0)
fps = 1000.0 / inference_ms if inference_ms else 0.0

print(f'Parameters       : {parameters:,}')
print(f'GFLOPs           : {gflops:.4f}' if gflops is not None else 'GFLOPs           : API không trả về giá trị')
print(f'Model size (MB)  : {model_size_mb:.2f}')
print(f'Inference (ms)   : {inference_ms:.2f}')
print(f'FPS              : {fps:.2f}')

## 12. Summary của E1

In [ ]:
summary = pd.DataFrame([{
    'Model': 'YOLO26s-P2',
    'Precision': validation_summary['Precision'],
    'Recall': validation_summary['Recall'],
    'F1': validation_summary['F1'],
    'mAP50': validation_summary['mAP50'],
    'mAP50-95': validation_summary['mAP50-95'],
    'NoHelmet_Precision': no_helmet_metrics['Precision'],
    'NoHelmet_Recall': no_helmet_metrics['Recall'],
    'NoHelmet_F1': no_helmet_metrics['F1'],
    'NoHelmet_AP50': no_helmet_metrics['AP50'],
    'NoHelmet_AP50_95': no_helmet_metrics['AP50-95'],
    'Parameters': parameters,
    'GFLOPs': gflops,
    'Inference_ms': inference_ms,
    'FPS': fps,
}])

display(summary)

## 13. So sánh tự động với baseline

Cell này load trực tiếp best.pt của baseline và E1, sau đó chạy cả hai trên cùng split=`val`, imgsz=640, device=0 để tạo bảng metric tổng và metric riêng của `rider_without_helmet`.

In [ ]:
# Comparison E1 chỉ dùng split='val'; không đọc results.csv để thay thế validation trực tiếp.
if not BASELINE_BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy baseline best.pt: {BASELINE_BEST_MODEL_PATH}')
if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f'Không tìm thấy YOLO26s-P2 best.pt: {BEST_MODEL_PATH}')

baseline_comparison_model = YOLO(BASELINE_BEST_MODEL_PATH)
p2_comparison_model = YOLO(BEST_MODEL_PATH)

baseline_comparison_metrics = baseline_comparison_model.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=640,
    device=0,
    plots=False,
)
p2_comparison_metrics = p2_comparison_model.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=640,
    device=0,
    plots=False,
)

baseline_comparison_summary = extract_metrics(baseline_comparison_metrics, 'YOLO26s baseline comparison')
p2_comparison_summary = extract_metrics(p2_comparison_metrics, 'YOLO26s-P2 comparison')

def comparison_row(model_name, summary_values):
    class_values = summary_values['by_class']['rider_without_helmet']
    return {
        'Model': model_name,
        'Precision': summary_values['Precision'],
        'Recall': summary_values['Recall'],
        'F1': summary_values['F1'],
        'mAP50': summary_values['mAP50'],
        'mAP50-95': summary_values['mAP50-95'],
        'rider_without_helmet Precision': class_values['Precision'],
        'rider_without_helmet Recall': class_values['Recall'],
        'rider_without_helmet F1': class_values['F1'],
        'rider_without_helmet AP50': class_values['AP50'],
        'rider_without_helmet AP50-95': class_values['AP50-95'],
    }

comparison = pd.DataFrame([
    comparison_row('YOLO26s', baseline_comparison_summary),
    comparison_row('YOLO26s-P2', p2_comparison_summary),
])
display(comparison)

## 14. Preflight audit và hướng dẫn chạy

Chạy các cell theo thứ tự từ đầu đến cuối trên Google Colab. E1 không chạy test set; test chỉ được dùng sau khi đã chốt Proposed Model cuối cùng.

In [ ]:
# Các assertion cuối notebook giúp tránh nhầm baseline/checkpoint/output.
assert MODEL_NAME == 'yolo26s.pt'
assert P2_MODEL_CONFIG == 'yolo26s-p2.yaml'
assert EXPERIMENT_NAME == 'helmet_yolo26s_p2_dataset2'
assert EXPERIMENT_NAME != BASELINE_EXPERIMENT_NAME
assert DATA_YAML_PATH.exists()
assert BEST_MODEL_PATH == OUTPUT_DIR / EXPERIMENT_NAME / 'weights' / 'best.pt'
assert BASELINE_EXPERIMENT_NAME not in str(BEST_MODEL_PATH)
assert getattr(best_model, 'task', None) == 'detect'
assert test_metrics is None and test_summary is None
expected_comparison_columns = [
    'Model', 'Precision', 'Recall', 'F1', 'mAP50', 'mAP50-95',
    'rider_without_helmet Precision', 'rider_without_helmet Recall',
    'rider_without_helmet F1', 'rider_without_helmet AP50',
    'rider_without_helmet AP50-95',
]
assert list(comparison.columns) == expected_comparison_columns
assert set(comparison['Model']) == {'YOLO26s', 'YOLO26s-P2'}

checkpoint_text = str(BEST_MODEL_PATH).lower()
for forbidden_token in ('yolo26n', 'yolo11n', 'yolo11s'):
    assert forbidden_token not in checkpoint_text, f'Checkpoint path chứa model không được phép: {forbidden_token}'

expected_plot_names = {filename for filename, _ in PLOT_SPECS}
print('Audit passed.')
print(f'Expected plot filenames: {sorted(expected_plot_names)}')
print(f'E1 best checkpoint: {BEST_MODEL_PATH}')
print(f'Baseline checkpoint remains separate: {BASELINE_BEST_MODEL_PATH}')

### Tóm tắt thay đổi so với `Train3.ipynb`

1. Cell model được đổi từ fine-tune trực tiếp `yolo26s.pt` sang `YOLO('yolo26s-p2.yaml')` rồi `model.load('yolo26s.pt')`; đây là khác biệt architecture duy nhất.
2. Dataset path, 3 class, cách setup Colab và protocol benchmark được giữ nguyên; output đổi sang `helmet_yolo26s_p2_dataset2`.
3. Checkpoint được load đúng từ `weights/best.pt` của E1.
4. Bổ sung metric tổng và theo class, riêng `rider_without_helmet`, validation comparison trực tiếp giữa hai best.pt, confusion-matrix error analysis, complexity/FPS, prediction examples trên validation, và summary.
5. Notebook không tạo YAML custom. Nếu phiên bản Ultralytics không có `yolo26s-p2.yaml`, notebook dừng rõ ràng thay vì tự thay architecture.